In [3]:
import qiskit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler.passes import SetLayout, TrivialLayout, CheckGateDirection, GatesInBasis
from qiskit.transpiler import PassManager, StagedPassManager

from qiskit.visualization import plot_histogram
from qiskit import QuantumRegister,QuantumCircuit,ClassicalRegister
from qiskit_ibm_runtime.fake_provider import FakeKingston

import matplotlib.pyplot as plt

### Feladat 1: Fulladder elkészítése



In [ ]:


full_adder = qiskit.QuantumCircuit(4)





sampler = qiskit.primitives.StatevectorSampler()
j1 = sampler.run([full_adder])
res_1 = j1.result()
plot_histogram(res_1[0].data.meas.get_counts())


### Feladat 2: Kézi layout elkészítése

![](figs/qc_bad.jpg)


In [ ]:

backend = FakeKingston()
qc_start = QuantumCircuit()
layout = []



pass_manager = StagedPassManager([ 'layout'],

        layout = PassManager([ SetLayout(layout)]),


    )
qc_final = pass_manager.run(qc_start)
qc_final.draw()

### Feladat 3: Kézi routing

![](figs/feladat.png)




In [ ]:
from qiskit.transpiler import CouplingMap

backend = FakeKingston()
qc_start = QuantumCircuit(4)


c_map = []
num_qubits = 7
for i in range(num_qubits-1):
    c_map.append([i,i+1])
    c_map.append([i+1,i])

custom_coupling = CouplingMap(c_map)

# Initialize GenericBackendV2 with the custom coupling map and number of qubits
basis_gates = ['cx', 'id', 'h','swap']
backend = GenericBackendV2(
    num_qubits=num_qubits,
    coupling_map=custom_coupling,
    basis_gates=basis_gates
)

pass_manager = StagedPassManager(['init'],
                                 init = PassManager([TrivialLayout(custom_coupling)])
                                 )

qc_final = pass_manager.run(qc_start)

test_pass_manager = StagedPassManager(['init'],
                                 init = PassManager([CheckGateDirection(custom_coupling),
                                                     GatesInBasis(basis_gates)

                                                     ])
                                 )

test_pass_manager.run(qc_final)

values = test_pass_manager.property_set

if not values["is_direction_mapped"] or values["all_getes_in_basis"]:
    print("Error, circuit is wrong :c")
else:
    job = backend.run(qc_final)
    result = job.result()
    counts = result.get_counts()
plot_histogram(counts)


### Feladat 4: SABRE számolása

$$H_{dist}=\sum_{g\in F}D[\pi(gate.q1),\pi(gate.q2)]$$

![](figs/feladat.png)

![](figs/feladat_qc.png)